Load Gold table

In [0]:
df_gold = spark.table("retail_project.gold.sales_features")

print("Rows:", df_gold.count())
df_gold.printSchema()

Convert to pandas

In [0]:
pdf_gold = df_gold.toPandas()
print(pdf_gold.shape)
pdf_gold.head()

Prepare features and target

In [0]:
feature_cols = [
    "Store", "DayOfWeek", "Open", "Promo", "SchoolHoliday", "IsStateHoliday",
    "Year", "Month", "WeekOfYear", "Sales_Lag1", "Sales_Lag7", "Sales_RollingAvg7",
    "TemperatureMean", "PrecipitationSum", "CompetitionDistance",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2", "Promo2SinceWeek", "Promo2SinceYear",
    "StoreType_Index", "Assortment_Index", "StateHoliday_Index"
]
target_col = "Sales"

X = pdf_gold[feature_cols]
y = pdf_gold[target_col]

print(X.shape, y.shape)
X.head()

Train/test split

In [0]:
pdf_gold_sorted = pdf_gold.sort_values("Date")
split_idx = int(len(pdf_gold_sorted) * 0.85)  # last 15% as test, roughly time-ordered

train_pdf = pdf_gold_sorted.iloc[:split_idx]
test_pdf = pdf_gold_sorted.iloc[split_idx:]

X_train, y_train = train_pdf[feature_cols], train_pdf[target_col]
X_test, y_test = test_pdf[feature_cols], test_pdf[target_col]

print("Train:", X_train.shape, "Test:", X_test.shape)

Install XGBoost

In [0]:
%pip install xgboost

Train the model

### Train XGBoost model (iteration 1: all days included)
Initial approach trains on all rows including closed days. MAPE calculation
requires filtering to Sales > 0 to avoid division-by-zero errors.

In [0]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import numpy as np

model_xgb = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)

preds = model_xgb.predict(X_test)

mape = mean_absolute_percentage_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"XGBoost MAPE: {mape:.2%}")
print(f"XGBoost RMSE: {rmse:.2f}")

### Train XGBoost model (iteration 2: open days only)
Excluding closed days from training entirely (not just evaluation) for
consistency with the Prophet approach, and to avoid the model learning from
rows where Sales=0 by definition rather than genuine low demand.

**Final result:** MAPE 10.41%, RMSE 1080.59 (evaluated on Sales > 0 rows,
54 anomalous Open=1/Sales=0 rows excluded from MAPE due to division by zero).

In [0]:
# Only evaluate MAPE on days the store was actually open (Sales > 0)
mask = y_test > 0

mape = mean_absolute_percentage_error(y_test[mask], preds[mask])
rmse = np.sqrt(mean_squared_error(y_test, preds))  # RMSE is fine to keep on all rows

print(f"XGBoost MAPE (open days only): {mape:.2%}")
print(f"XGBoost RMSE (all days): {rmse:.2f}")

In [0]:
# Retrain excluding closed days
train_open = train_pdf[train_pdf["Open"] == 1]
test_open = test_pdf[test_pdf["Open"] == 1]

X_train_open, y_train_open = train_open[feature_cols], train_open[target_col]
X_test_open, y_test_open = test_open[feature_cols], test_open[target_col]

model_xgb_v2 = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
model_xgb_v2.fit(X_train_open, y_train_open)

preds_v2 = model_xgb_v2.predict(X_test_open)

mape_v2 = mean_absolute_percentage_error(y_test_open, preds_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_test_open, preds_v2))

print(f"XGBoost MAPE (trained + evaluated on open days only): {mape_v2:.2%}")
print(f"XGBoost RMSE (trained + evaluated on open days only): {rmse_v2:.2f}")

In [0]:
zero_sales_but_open = pdf_gold[(pdf_gold["Open"] == 1) & (pdf_gold["Sales"] == 0)]
print("Rows with Open=1 but Sales=0:", len(zero_sales_but_open))
zero_sales_but_open[["Store", "Date", "Sales", "Open"]].head(10)

In [0]:
mask = y_test_open > 0

mape_v2 = mean_absolute_percentage_error(y_test_open[mask], preds_v2[mask])
rmse_v2 = np.sqrt(mean_squared_error(y_test_open, preds_v2))

print(f"XGBoost MAPE (Sales > 0 only): {mape_v2:.2%}")
print(f"XGBoost RMSE (all open-day rows): {rmse_v2:.2f}")

MLflow integration

In [0]:
from mlflow.models.signature import infer_signature

# Infer the signature from your training data and predictions
signature = infer_signature(X_train_open, preds_v2)

mlflow.set_experiment("/Shared/retail_demand_forecasting")

with mlflow.start_run(run_name="xgboost_all_stores_v2"):
    mlflow.log_param("model_type", "XGBRegressor")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("features", "all_stores_open_days_only")
    mlflow.log_metric("mape", mape_v2)
    mlflow.log_metric("rmse", rmse_v2)
    mlflow.xgboost.log_model(
        model_xgb_v2,
        "model",
        signature=signature,
        input_example=X_train_open.head(5)
    )
    run_id = mlflow.active_run().info.run_id

print(f"Model logged with signature. Run ID: {run_id}")

Register the best model

In [0]:
model_uri = f"runs:/{run_id}/model"
registered_model = mlflow.register_model(model_uri, "retail_project.gold.retail_demand_xgboost")

print(f"Registered model: {registered_model.name}, version: {registered_model.version}")

Promote as champion

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="retail_project.gold.retail_demand_xgboost",
    alias="champion",
    version=registered_model.version
)

print("Model aliased as 'champion' (production-designated) version.")

### Summary

Trained an XGBoost model across all 1,115 stores simultaneously (unlike Prophet,
which required a separate model per store).

**Final result:** MAPE 10.41%, RMSE 1080.59 (evaluated on Sales > 0 rows only;
54 anomalous Open=1/Sales=0 rows excluded to avoid division-by-zero in MAPE).

Model registered in Unity Catalog as `retail_project.gold.retail_demand_xgboost`
(version 1), tagged with the `champion` alias to mark it as the production-
designated model.